# ಪಾಠ 02 - ಮೈಕ್ರೋಸಾಫ್ಟ್ ಏಜೆಂಟ್ ಫ್ರೇಮ್ವರ್ಕ್ ಅನ್ವೇಷಣೆ

**ಮೈಕ್ರೋಸಾಫ್ಟ್ ಏಜೆಂಟ್ ಫ್ರೇಮ್ವರ್ಕ್ (MAF)** ಎಂಬುದು AI ಏಜೆಂಟ್‌ಗಳನ್ನು ನಿರ್ಮಿಸಲು ಏಕೀಕೃತ ಫ್ರೇಮ್ವರ್ಕ್ ಆಗಿದೆ. ಇದು ನಾಲ್ಕು ಪ್ರಮುಖ ನಿರ್ಮಾಣ ಅದುಗಳನ್ನು ಹೊಂದಿರುವ ಶುಚಿ, ಸಂಯೋಜಿಸಬಹುದಾದ ವಾಸ್ತುಶಿಲ್ಪವನ್ನು ಒದಗಿಸುತ್ತದೆ:

- **ಗ್ರಾಹಕ** – AI ಮಾದರಿ ಎಂಡ್‌ಪಾಯಿಂಟ್‌ಗೆ ಸಂಪರ್ಕಿಸುತ್ತಿದ್ದು ಸಂವಹನವನ್ನು ನಿರ್ವಹಿಸುತ್ತದೆ  
- **ಏಜೆಂಟ್** – ಸೂಚನೆಗಳು ಮತ್ತು ಸಾಧನ ವ್ಯಾಖ್ಯಾನಗಳೊಂದಿಗೆ ಗ್ರಾಹಕವನ್ನು ಯೂಜ್ ಮಾಡುತ್ತದೆ  
- **ಉಪಕರಣಗಳು** – ಮಾದರಿ ಕರೆ ಮಾಡಬಹುದಾದ ಕাস্টಮ್ ಕಾರ್ಯಾಚರಣೆಗಳೊಂದಿಗೆ ಏಜೆಂಟ್ ಸಾಮರ್ಥ್ಯಗಳನ್ನು ವಿಸ್ತರಿಸುತ್ತವೆ  
- **ಹೊಂದಿಕೆ** – ಬಹು-ತಿರುವು ಸಂವಹನಗಳಿಗೆ ಸಂಭಾಷಣೆ ಇತಿಹಾಸವನ್ನು ಕಾಪಾಡುತ್ತದೆ

ಈ ಪಾಠದಲ್ಲಿ, ನಾವು ಈ ಕಲ್ಪನೆಗಳನ್ನು ಬಳಸಿ ಗಮ್ಯಸ್ಥಲ ಲಭ್ಯತೆ ಪರಿಶೀಲಿಸುವ **ಪ್ರಯಾಣ ಬುಕ್ಕಿಂಗ್ ಏಜೆಂಟ್** ಅನ್ನು ನಿರ್ಮಿಸುತ್ತೇವೆ.


## ಸೆಟಪ್


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## ಏಜೆಂಟ್ ಫ್ರೇಂವರ್ಕ್ ವಾಸ್ತುಶಿಲ್ಪವನ್ನು ಅರ್ಥಮಾಡಿಕೊಳ್ಳುವುದು

Microsoft Agent Framework ಒಂದು ಪದರಿತ ವಾಸ್ತುಶಿಲ್ಪವನ್ನು ಅನುಸರಿಸುತ್ತದೆ:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **ಗ್ರಾಹಕ** – `AzureAIProjectAgentProvider` ಒಂದು Azure OpenAI ನಿಯೋಜನಿಗೆ ಸಂಪರ್ಕ ಹೊಂದಿಸುತ್ತದೆ. ಇದು ದೃಧೀಕರಣ, ವಿನಂತಿ ಸ್ವರೂಪೀಕರಣೆ ಮತ್ತು ಪ್ರತಿಕ್ರಿಯೆ ಪಾರ್ಸಿಂಗ್ ಅನ್ನು ನಿರ್ವಹಿಸುತ್ತದೆ.
2. **ಏಜೆಂಟ್** – ಗ್ರಾಹಕದಿಂದ `provider.create_agent()` ಮೂಲಕ ರಚಿಸಲಾಗುತ್ತದೆ, ಏಜೆಂಟ್ ಮಾದರಿ ಪ್ರವೇಶವನ್ನು ಸೂಚನೆಗಳೊಂದಿಗೆ (ಸಿಸ್ಟಮ್ ಪ್ರಾಂಪ್ಟ್) ಮತ್ತು ಸಾಧನಗಳೊಂದಿಗೆ ಸಂಯೋಜಿಸುತ್ತದೆ.
3. **ಸಾಧನಗಳು** – ಏಜೆಂಟ್ ಕಾರ್ಯಾಚರಣೆಗಳನ್ನು ನಿರ್ವಹಿಸಲು ಅಥವಾ ಡೇಟಾವನ್ನು ಪಡೆಯಲು ಕರೆ ಮಾಡಬಹುದಾದ `@tool` ಅಲಂಕರಣಗೊಂಡ ಪೈಥಾನ್ ಫังก್ಷನ್ಗಳು.
4. **ಸತ್ರ** – `AgentSession` ಆಬ್ಜೆಕ್ಟ್ (ಏಜೆಂಟ್ ಅನ್ನು `agent.create_session()` ಮೂಲಕ ಸೃಷ್ಟಿಸಲಾಗಿದೆ) ಸಂಭಾಷಣೆ ಇತಿಹಾಸವನ್ನು ಸಂಗ್ರಹಿಸುತ್ತದೆ, ಏಜೆಂಟ್ ಹಿಂದಿನ ಸಂದರ್ಭವನ್ನು ನೆನಪಿಡುವ ಬಹು-ತಿರುಗು ಸಂವಾದಕ್ಕೆ ಅನುಮತಿಸುತ್ತದೆ.

ಪ್ರತಿ ಪದರವನ್ನು ಕ್ರಮವಾಗಿ ನಿರ್ಮಿಸೋಣ.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool ಅಲೆಂಕಾರಕದೊಂದಿಗೆ ಉಪಕರಣಗಳನ್ನು ಸೇರಿಸುವುದು

ಉಪಕರಣಗಳು ಪ್ರತಿನಿಧಿಗಳಿಗೆ ಪಠ್ಯವನ್ನು ಸೃಷ್ಟಿಸುವುದನ್ನು ಹೊರತುಪಡಿಸಿ ಕ್ರಮಗಳನ್ನು ಕೈಗೊಳ್ಳಲು ಸಾ ಮರ್ಥ್ಯವನ್ನು ಒದಗಿಸುತ್ತವೆ. `@tool` ಅಲೆಂಕಾರಕವು ಸಾಮಾನ್ಯ ಪೈಥಾನ್ ಫಂಕ್ಷನ್ ಅನ್ನು ಪ್ರತಿನಿಧಿಯು ಕರೆದುಕೊಳ್ಳಬಹುದಾದ ಏನಾದರೂ ಆಗಿ ಪರಿವರ್ತಿಸುತ್ತದೆ.

ಪ್ರಮੁੱਖ ವಿಚಾರಗಳು:
- ಮಾದರಿ ಪ್ರತಿ ಪರಾಮಿತಿ ಅನ್ನು ಅರ್ಥಮಾಡಿಕೊಳ್ಳುವಂತೆ ಮಾಡಲು `Annotated[type, "description"]` ಅನ್ನು ಬಳಸಿ.
- ಡಾಕ್‌ಸ್ಟ್ರಿಂಗ್ ಆ ಉಪಕರಣದ ವಿವರಣೆಯಾಗಿ ಮಾದರಿಗೆ ಕಾಣಿಸುತ್ತದೆ.
- `approval_mode="never_require"` ಅಂದರೆ ಉಪಕರಣವು ಬಳಕೆದಾರ ದೃಢೀಕರಣವಿಲ್ಲದೆ ಸ್ವಯಂ ಚಾಲನೆಗೊಳ್ಳುತ್ತದೆ.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## ಟೂಲ್ಸ್‌ ಸಹಿತ ಏಜೆಂಟ್ ರಚನೆ

ಈಗ ನಾವು ಕ್ಲೈಂಟ್, ಸೂಚನೆಗಳು ಮತ್ತು ಟೂಲ್ಸ್ ಅನ್ನು ಏಜೆಂಟ್‌ನಲ್ಲಿ ಸಂಯೋಜಿಸುತ್ತೇವೆ. `ಸೂಚನೆಗಳು` ವ್ಯವಸ್ಥೆ ಪ್ರಾಂಪ್ಟ್ ಆಗಿವೆ — ಅವು ಏಜೆಂಟ್‌ನ ವ್ಯಕ್ತಿತ್ವ ಮತ್ತು ವರ್ತನೆಯನ್ನ ನಿರ್ಧರಿಸುತ್ತವೆ.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## ಸೆಷನ್‌ಗಳೊಂದಿಗೆ ಬಹು-ತಿರುವು ಸಂವಾದಗಳು

`AgentSession` (`agent.create_session()` ಮೂಲಕ ಸೃಷ್ಟಿಸಲಾದ) ಸಂವಾದದಲ್ಲಿನ ಎಲ್ಲಾ ಸಂದೇಶಗಳನ್ನು ಅನುಸರಿಸುತ್ತದೆ. ಪ್ರತಿಯೊಂದು `agent.run()` ಕರೆಗೆ ಅದೇ ಸೆಷನ್ ಅನ್ನು ಪಾಸ್ ಮಾಡುವುದರಿಂದ, ಏಜೆಂಟ್‌ಗೆ ಸಂಪೂರ್ಣ ಸಂವಾದ ಇತಿಹಾಸವನ್ನು ಪ್ರವೇಶಿಸುವ ಅವಕಾಶ ದೊರೆಯುತ್ತದೆ ಮತ್ತು ಹಳೆಯ ಸಂದೇಶಗಳನ್ನು ಉಲ್ಲೇಖಿಸಬಹುದು.

ನಮ್ಮ ಕೈಗೊಳ್ಳುವಿಕೆಯ ಪರಿಶೀಲಕವನ್ನು ಪ್ರತಿ ತಿರುವಾಗೂ ಕರೆ ಮಾಡಲು ಏಜೆಂಟ್‌ಗೂ ಸಹಾಯ ಮಾಡಲು `tools=[check_destination_availability]` ಅನ್ನು ಪಾಸ್ ಮಾಡುತ್ತೇವೆ.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು Microsoft Agent Framework ರ ನಾಲ್ಕು ತೃಣಾಧಾರಗಳನ್ನು ಅನ್ವೇಷಿಸಿದ್ದೀರಿ:

| ಪರಿಕಲ್ಪನೆ | ನೀವು ಕಲಿತದ್ದು |
|---------|------------------|
| **ಕ್ಲೈಂಟ್** | `AzureAIProjectAgentProvider` ಕ್ರೆಡೆನ್ಷಿಯಲ್-ಆಧಾರಿತ ಪ್ರಾಮಾಣೀಕರಣದೊಂದಿಗೆ Azure OpenAI ಗೆ ಸಂಪರ್ಕಿಸುತ್ತದೆ |
| **ಏಜೆಂಟ್** | `provider.create_agent()` ಮಾದರಿ ಸಂಪರ್ಕವನ್ನು ಸೂಚನೆಗಳು ಮತ್ತು ಹೆಸರಿನೊಂದಿಗೆ ಜೋಡಿಸುತ್ತದೆ |
| **ಉಪಕರಣಗಳು** | `@tool` ಡೆಕೋರೇಟರ್ ಏಜೆಂಟ್ ಕರೆ ಮಾಡಬೇಕಾದ Python ಫಂಕ್ಷನ್‌ಗಳನ್ನು ಬಹಿರಂಗಪಡಿಸುತ್ತದೆ |
| **ಸೆಷನ್** | `agent.create_session()` ಹಲವಾರು ತಿರುವುಗಳ ನಡುವೆ ಸಂಭಾಷಣೆ ಇತಿಹಾಸವನ್ನು ಕಾಯ್ದಿರಿಸುತ್ತದೆ |

ಈ ಕಟ್ಟಡಘಟಕಗಳು ಸೇರಿಕೊಂಡು ನೈಸರ್ಗಿಕ ಸಂಭಾಷಣೆಗಳನ್ನು ನಡೆಸಬಹುದಾದ, ಬಾಹ್ಯ ಫಂಕ್ಷನ್‌ಗಳನ್ನೂ ಕರೆ ಮಾಡಬಹುದಾದ ಮತ್ತು_context_ ಕಾಯ್ದುಕೊಳ್ಳುವ ಏಜೆಂಟ್‌ಗಳನ್ನು ರಚಿಸುತ್ತವೆ — ನಂತರದ ಪಾಠಗಳಲ್ಲಿ ಅಭಿವೃದ್ಧಿಪಡಿಸಲಾದ ಏಜೆಂಟ್ ಮಾದರಿಗಳ ಮೇಲ್ಪಡೆ.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಬೀಳ್ಕೆ**:
ಈ ದಸ್ತಾವೇಜನ್ನು AI ಭಾಷಾಂತರ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಭಾಷಾಂತರಿಸಲಾಗಿದೆ. ನಾವು ಸರಿಯಾದ ಭಾಷಾಂತರದಿಗಾಗಿ ಪ್ರಯತ್ನಿಸುವುದಾದರೂ, ಸ್ವಯಂರಾಮಭರಿತ ಭಾಷಾಂತರದಲ್ಲಿ ತಪ್ಪುಗಳು ಅಥವಾ ಅಸತ್ಯತೆಗಳಾಗಿರಬಹುದು என்பதை ಜ್ಞಾಪಕರಾಗಿರಲಿ. ಮೂಲ ದಸ್ತಾವೇಜಿನ ಮೂಲಭಾಷೆಯ ಆವೃತ್ತಿಯನ್ನು ಅಧಿಕೃತ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ ವೃತ್ತಿಪರ ಮಾನವ ಭಾಷಾಂತರವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಭಾಷಾಂತರ ಬಳಕೆಯಿಂದ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಮಾಡಿಕೊಳ್ಳಿಕೆಗಳು ಅಥವಾ ತಪ್ಪು ವ್ಯಕ್ತಪಡಿಸುವಿಕೆಗಳಿಗೆ ನಾವು ಜವಾಬ್ದಾರರಾಗುವುದಿಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
